General structure and idea:
- The function filter_for_mars_wsa(cmes) receives a list of CME events in the JSON format from the DONKI API. This translates to a Python dictionary, so each CME event is a dictionary.
- For each CME dict, it extracts the list wsaAnalyses.
- Then, it scans every entry in this list. It then checks whether the word "Mars" appears in any of the entries' targetList.
- If Mars is predicted, it gets the first CME analysis and reads its speed.
- A filter on the velocity is applied. It only keeps the events whose speed is above the threshold.
- Finally it builds the alters. For each qualified CME, it collects a tuple with three values and appends it to the alerts list.
- The output returns a list of tuples per CME that WSA predicts will hit Mars at enough speed.


In [7]:
import requests
import time
import logging
from datetime import datetime, timedelta

from datetime import datetime, timezone

end = datetime.now(timezone.utc)

In [8]:
# Configuration 
API_KEY = 'uSWNbLjTbxucLFPWp3es0mZILwWkq0tBzsxgfDu1'  # Used an API key from https://api.nasa.gov
MIN_SPEED = 1000  # km/s # Speed condition imposed
CHECK_INTERVAL = 60 * 10  # seconds between consecutive checks (10 min)
DAYS_LOOKBACK = 3  

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [ ]:

# Downloads the results from the DONKI website for the CMEs for a specific time interval.

def fetch_cmes(start_date: str, end_date: str):
    """
    Checks DONKI's API REST for CME events between the start_date and the end_date. Returns a list of 
    dictionaries (JSON).
    """
    url = 'https://api.nasa.gov/DONKI/CME'
    params = {
        'startDate': start_date,
        'endDate': end_date,
        'catalog': 'M2M_CATALOG',
        'api_key': API_KEY
    }
    response = requests.get(url, params=params)
    response.raise_for_status() # function that checks the state of the HTTPS code for errors 
    return response.json()

In [ ]:
# Filters the list of CMEs that the API returns and only returns those which meet two criteria: the WSA model
# predicts and impact on Mars and the eyection speed is at least MIN_SPEED.
# dict.get(key, default_value) is the structure used below.
def filter_for_mars_wsa(cmes):
    """
    Filters CMEs where the WSA model predicts an impact on Mars and whose eyection speed is >= MIN_SPEED.
    Returns lists of tuples, each tuple having 3 values: its id (cmeID - a string), the speed (number) and 
    its corresponding link (linkDetails - a string).
    """
    alerts = []
    for cme in cmes: #cme is a dictionary that represents a CME event as retrieved from DONKI's API
        # Checks the WSA lists to see if Mars appears on 'targetList'
        wsa_analyses = cme.get('wsaAnalyses', []) # wsa_analyses is the key from the dictionary
        if any('Mars' in target for analysis in wsa_analyses for target in analysis.get('targetList', [])):
            # Obtains eyection speed (from the first CME analysis)
            speed = cme.get('cmeAnalyses', [{}])[0].get('speed')
            if speed and speed >= MIN_SPEED:
                cme_id = cme.get('activityID')
                link = f"https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/CME/{cme_id}"
                alerts.append((cme_id, speed, link))
    return alerts

In [10]:
def main():
    while True:
        end = datetime.now(timezone.utc)
        start = end - timedelta(days=DAYS_LOOKBACK)
        start_str = start.strftime('%Y-%m-%d')
        end_str = end.strftime('%Y-%m-%d')

        try:
            cmes = fetch_cmes(start_str, end_str)
            mars_alerts = filter_for_mars_wsa(cmes)
            if mars_alerts:
                for cme_id, speed, link in mars_alerts:
                    logging.info(f" CME WSA towards Mars detected. ID: {cme_id} | Speed: {speed} km/s | Details: {link}")
                    # Can call functions here to download plasma parameters and create plots.
            else:
                logging.info("No new CMEs with WSA prediction towards Mars in the selected time interval.")
        except Exception as e:
            logging.error(f"Error consulting the API: {e}")

        time.sleep(CHECK_INTERVAL)


if __name__ == '__main__':
    main()


2025-07-06 21:20:28 INFO: No new CMEs with WSA prediction towards Marte in the selected interval.


KeyboardInterrupt: 